# 競輪予想AI（設計書版）— Google Colab 用

上のセルから順に ▶ を押していけば動きます。**お金は一切使いません**（すべて仮想購入）。

| 手順 | やること |
|---|---|
| 1 | Google ドライブをつなぐ（データとExcelを保存する場所） |
| 2 | プログラムを取ってくる |
| 3 | 架空データでお試し（動作確認） |
| 4 | 本物のデータで検証（データが用意できてから） |

## 1. Google ドライブをつなぐ
許可の画面が出たら「許可」を押します。ドライブに `keirin_ai_data` フォルダが作られます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/keirin_ai_data'
for sub in ['inbox', 'output', 'data', 'logs']:
    os.makedirs(f'{WORK}/{sub}', exist_ok=True)
print('保存先:', WORK)

## 2. プログラムを取ってくる
GitHub のリポジトリが **非公開** の場合は、この方法では取れません。その場合は GitHub の「Code → Download ZIP」で落とした ZIP を
ドライブの `keirin_ai_data` に入れて、下のセルの `USE_ZIP = True` にしてください。

In [ ]:
USE_ZIP = False
BRANCH = 'claude/keirin-prediction-ai-47mdd4'

%cd /content
if USE_ZIP:
    import glob
    z = sorted(glob.glob(f'{WORK}/*.zip'))[-1]
    !rm -rf /content/uni && mkdir -p /content/uni && unzip -q -o "$z" -d /content/uni_zip && cp -r /content/uni_zip/*/keirin_ai /content/uni/
else:
    !rm -rf /content/uni && git clone -q -b $BRANCH https://github.com/uni620/uni.git /content/uni
%cd /content/uni/keirin_ai
!pip install -q -r requirements.txt
print('準備OK')

設定ファイルを作ります（データベースとExcelの保存先をドライブにします）。

In [ ]:
cfg = open('config.example.toml', encoding='utf-8').read()
cfg = cfg.replace('db = "data/keirin.sqlite"', f'db = "{WORK}/data/keirin.sqlite"')
cfg = cfg.replace('excel = "output/回収率レポート.xlsx"', f'excel = "{WORK}/output/検証レポート.xlsx"')
cfg = cfg.replace('log = "logs/keirin.log"', f'log = "{WORK}/logs/keirin.log"')
cfg = cfg.replace('name = "sample"', 'name = "csv"')
cfg = cfg.replace('csv_inbox = "inbox"', f'csv_inbox = "{WORK}/inbox"')
cfg = cfg.replace('race_names_csv = "race_names.csv"', f'race_names_csv = "{WORK}/race_names.csv"')
open('config.toml', 'w', encoding='utf-8').write(cfg)
print('config.toml を作りました')

## 3. 架空データでお試し（1分ほど）
架空の選手・架空のレース（約4000レース）で、**特徴量 → 学習 → 検証 → Excel** まで一通り動かします。

⚠️ 架空データの数字は、**どの予想方法が良いかの判断には使えません**（動作確認用）。

In [ ]:
!python -m keirin demo

In [ ]:
# 結果を表で見る
import json, pandas as pd
ev = json.load(open('data/demo_evaluation.json', encoding='utf-8'))
print(f"学習 {ev['train_races']}レース / 検証 {ev['test_races']}レース（{ev['cut_date']} 以降）")
display(pd.DataFrame(ev['table']))
print('LightGBM が重視した列（上位10）')
display(pd.DataFrame(ev['importance']['LightGBM']).head(10))

In [ ]:
# Excel をパソコンに保存する
from google.colab import files
files.download('output/デモ_検証レポート.xlsx')

## 4. 本物のデータで検証
1. ドライブの `keirin_ai_data/inbox` に、`templates` フォルダと同じ形の CSV を入れる
   （`races*.csv` / `entries*.csv` / `odds*.csv` / `payouts*.csv`）
2. 下のセルを順に実行する

レース名から格を自動で決められなかったものは、Excel の「格が不明なレース名」に出ます。
`keirin_ai_data/race_names.csv`（列: `race_name,kaku`）に点数を書くと、次回から使われます。

In [ ]:
!python -m keirin import-csv

In [ ]:
# 古いレースで学習し、新しい約22%で試す（4500レースなら約1000レース）
# 締切前オッズが無いデータのときは、末尾に --allow-final-odds を付ける（回収率は甘くなる＝参考値）
!python -m keirin evaluate

In [ ]:
ev = json.load(open(f'{WORK}/data/evaluation.json', encoding='utf-8'))
display(pd.DataFrame(ev['table']))
display(pd.DataFrame(ev['importance']['LightGBM']).head(10))
files.download(f'{WORK}/output/検証レポート.xlsx')